# Keller-Segel Slime Mold Chemotaxis Simulation

Simulating *Dictyostelium* chemotaxis toward an external chemical attractant using the Keller-Segel model with Monod growth kinetics, solved with the finite volume method (FiPy) on a 2D rectangular domain.

**Coupled PDEs:**
- Cell density: $\partial\rho/\partial t = D_\rho \nabla^2\rho - \chi \nabla\cdot(\rho \nabla c) + \frac{\mu_{\max}}{Y} c \, \rho (1 - \rho/\rho_{\max})$
- Chemoattractant (quasi-steady-state): $0 = D_c \nabla^2 c - \beta c - \mu_{\max} c \, \rho$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

from keller_segel import KellerSegelParams, run_simulation

## Parameter Configuration

| Parameter | Value | Description |
|-----------|-------|-------------|
| `D_rho` | 1e-3 | Cell diffusion coefficient (random motility) |
| `chi` | 5e-3 | Chemotactic sensitivity (migration strength toward attractant) |
| `mu_max` | 0.02 | Maximum specific growth rate (Monod kinetics) |
| `Y` | 0.5 | Yield coefficient (substrate consumed per unit biomass produced) |
| `rho_max` | 5.0 | Carrying capacity (prevents finite-time blow-up) |
| `D_c` | 1e-2 | Chemical diffusion coefficient |
| `beta` | 0.05 | Chemical decay rate |
| `c_boundary` | 1.0 | Dirichlet BC value (fixed attractant at boundaries) |
| `dt` | 0.01 | Timestep for cell dynamics |
| `total_time` | 50.0 | Total simulation time |

In [ ]:
params = KellerSegelParams(
    Lx=10.0, Ly=10.0,
    nx=100, ny=100,
    D_rho=1e-3,
    chi=5e-3,
    mu_max=0.01,
    rho_max=5.0,
    D_c=1e-2,
    beta=0.05,
    c_boundary=1.0,
    dt=0.01,
    total_time=50.0,
    sweep_count=5,
    snapshot_interval=50,
)

print(f"Domain: {params.Lx} x {params.Ly}, Mesh: {params.nx} x {params.ny}")
print(f"dx = {params.dx}, dy = {params.dy}")
print(f"Total steps: {params.n_steps}, Snapshots: ~{params.n_steps // params.snapshot_interval}")
print(f"chi/D_rho ratio: {params.chi / params.D_rho:.1f} (chemotaxis vs diffusion strength)")

## Run Simulation

In [ ]:
result = run_simulation(params, progbar=True)

print(f"\nSimulation complete.")
print(f"Snapshots saved: {len(result.times)}")
print(f"Final total mass: {result.total_mass[-1]:.4f}")
print(f"Final max density: {result.max_density[-1]:.4f}")

## Static Snapshots

Cell density (top, inferno) and chemoattractant concentration (bottom, viridis) at three timepoints.

In [ ]:
n_snapshots = len(result.times)
indices = [0, n_snapshots // 2, n_snapshots - 1]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

rho_vmax = max(snap.max() for snap in result.rho_snapshots)
c_vmax = max(snap.max() for snap in result.c_snapshots)

for col, idx in enumerate(indices):
    t = result.times[idx]

    # Cell density
    im_rho = axes[0, col].imshow(
        result.rho_snapshots[idx],
        origin="lower",
        extent=[0, params.Lx, 0, params.Ly],
        cmap="inferno",
        vmin=0, vmax=rho_vmax,
    )
    axes[0, col].set_title(f"$\\rho$ at t = {t:.1f}")
    axes[0, col].set_xlabel("x")
    axes[0, col].set_ylabel("y")

    # Chemoattractant
    im_c = axes[1, col].imshow(
        result.c_snapshots[idx],
        origin="lower",
        extent=[0, params.Lx, 0, params.Ly],
        cmap="viridis",
        vmin=0, vmax=c_vmax,
    )
    axes[1, col].set_title(f"c at t = {t:.1f}")
    axes[1, col].set_xlabel("x")
    axes[1, col].set_ylabel("y")

fig.colorbar(im_rho, ax=axes[0, :], label="Cell density $\\rho$", shrink=0.8)
fig.colorbar(im_c, ax=axes[1, :], label="Attractant c", shrink=0.8)
fig.suptitle("Keller-Segel Chemotaxis: Snapshots", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("snapshots.png", dpi=150, bbox_inches="tight")
plt.show()

## Diagnostics: Total Mass and Max Density

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Total mass includes both snapshot and per-step recordings
# Use per-step data for smooth plots
steps = np.arange(len(result.total_mass))

ax1.plot(steps, result.total_mass, color="steelblue", linewidth=1)
ax1.set_xlabel("Recording index")
ax1.set_ylabel("Total mass $\\int \\rho \, dA$")
ax1.set_title("Total Cell Mass Over Time")
ax1.grid(True, alpha=0.3)

ax2.plot(steps, result.max_density, color="firebrick", linewidth=1)
ax2.axhline(y=params.rho_max, color="gray", linestyle="--", label=f"$\\rho_{{max}}$ = {params.rho_max}")
ax2.set_xlabel("Recording index")
ax2.set_ylabel("Max density $\\rho_{max}$")
ax2.set_title("Maximum Cell Density Over Time")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

## Animation: Cell Density and Chemoattractant

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

rho_vmax = max(snap.max() for snap in result.rho_snapshots)
c_vmax = max(snap.max() for snap in result.c_snapshots)

im1 = ax1.imshow(
    result.rho_snapshots[0], origin="lower",
    extent=[0, params.Lx, 0, params.Ly],
    cmap="inferno", vmin=0, vmax=rho_vmax,
)
ax1.set_title(f"$\\rho$ at t = 0.0")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
fig.colorbar(im1, ax=ax1, label="$\\rho$", shrink=0.8)

im2 = ax2.imshow(
    result.c_snapshots[0], origin="lower",
    extent=[0, params.Lx, 0, params.Ly],
    cmap="viridis", vmin=0, vmax=c_vmax,
)
ax2.set_title(f"c at t = 0.0")
ax2.set_xlabel("x")
ax2.set_ylabel("y")
fig.colorbar(im2, ax=ax2, label="c", shrink=0.8)

plt.tight_layout()


def animate(frame):
    im1.set_data(result.rho_snapshots[frame])
    im2.set_data(result.c_snapshots[frame])
    t = result.times[frame]
    ax1.set_title(f"$\\rho$ at t = {t:.1f}")
    ax2.set_title(f"c at t = {t:.1f}")
    return [im1, im2]


anim = animation.FuncAnimation(
    fig, animate,
    frames=len(result.times),
    interval=200,
    blit=True,
)

HTML(anim.to_html5_video())

## Analysis

### Aggregation Behavior
The simulation demonstrates the core Keller-Segel mechanism: cells initially concentrated at the center diffuse outward via random motility ($D_\rho$) while simultaneously migrating up the chemoattractant gradient ($\chi \nabla c$). Since the attractant is supplied at the boundaries (Dirichlet BCs) and decays in the interior ($\beta c$), a concentration gradient forms pointing outward from the center. Cells follow this gradient, accumulating near the boundaries.

### Monod Growth and Substrate Consumption
Cells consume substrate at rate $\mu_{\max} \cdot c \cdot \rho$ and grow at rate $\frac{\mu_{\max}}{Y} \cdot c \cdot \rho \cdot (1 - \rho/\rho_{\max})$. The yield coefficient $Y$ (substrate consumed per unit biomass) appears in the denominator — lower $Y$ means more efficient conversion of substrate to biomass. As cells accumulate near boundaries and consume the attractant, the local concentration drops, weakening both the growth signal and the chemotactic gradient. This creates a self-limiting feedback loop. The logistic cap $(1 - \rho/\rho_{\max})$ remains necessary because the Dirichlet BCs continuously supply fresh substrate at the boundaries, so substrate limitation alone cannot prevent unbounded density growth near the edges.

### Conservation
Total mass is *not* conserved due to the growth term. However, total mass should increase more slowly than with constant-rate logistic growth because growth is substrate-limited. As cells consume $c$, the growth rate diminishes — eventually the system may approach a quasi-equilibrium where consumption balances boundary supply.

### The $\chi / D_\rho$ Ratio
The ratio of chemotactic sensitivity to diffusion ($\chi / D_\rho = 5.0$ with default parameters) controls whether aggregation or dispersal dominates:
- **High ratio**: Strong chemotaxis relative to diffusion — cells form tight aggregates near attractant sources.
- **Low ratio**: Diffusion dominates — cells spread out nearly uniformly despite the chemical gradient.
- The logistic cap at $\rho_{\max}$ prevents finite-time blow-up, making the model well-posed for all parameter values.

### Numerical Stability
The `ExponentialConvectionTerm` (Scharfetter-Gummel scheme) is critical for stability when the Peclet number $Pe = \chi |\nabla c| \Delta x / D_\rho$ is large. Standard upwind or central differencing would produce spurious oscillations in this regime.